# Set up

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[1]))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from abc import ABC, abstractmethod

In [3]:
# from Part2_Classification.utils import *

## Constants

In [4]:
SEED = 42
LEARNING_RATE = 0.01
MAX_ITER = 10000

# Process data

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# STEP 1: LOAD PRE-CLEANED DATASET
# Loading the file we saved previously with medians already filled
df = pd.read_csv('dataset.csv')

# STEP 2: DEFINE FEATURES AND TARGET
# Splitting the dataframe into input features (X) and the label (y)
X = df.drop('Class', axis=1)
y = df['Class']

# STEP 3: PARTITION DATA INTO TRAIN AND TEST SETS
# Reserving 20% of the data for an unbiased evaluation
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 4: ENCODE CATEGORICAL LABELS
# Transforming bean names into numerical integers (0, 1, 2...)
target_le = LabelEncoder()
y_train = target_le.fit_transform(y_train)
y_test = target_le.transform(y_test)

# STEP 5: FEATURE STANDARDIZATION
# Scaling features to have a mean of 0 and variance of 1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# STEP 6: VERIFICATION OF FINAL ARRAYS
print("--- Final Data Ready ---")
print(f"Training Features Shape: {X_train_scaled.shape}")
print(f"Testing Features Shape: {X_test_scaled.shape}")
print(f"Classes to Predict: {target_le.classes_}")

--- Final Data Ready ---
Training Features Shape: (10888, 16)
Testing Features Shape: (2723, 16)
Classes to Predict: ['BARBUNYA' 'BOMBAY' 'CALI' 'DERMASON' 'HOROZ' 'SEKER' 'SIRA']


In [6]:
print(X_test_scaled)
print(X_train_scaled)


[[-0.36296    -0.52964611 -0.69695652 ...  1.15089146  1.54182176
   0.98769508]
 [ 0.52991536  1.10200072  0.59473961 ... -0.63053967 -0.1045282
  -2.01294548]
 [-0.517459   -0.68228083 -0.83494449 ...  1.26116851  1.38748903
   0.7233148 ]
 ...
 [-0.84293047 -1.08464304 -1.08258587 ...  1.17729546  0.56974941
   0.65879757]
 [ 0.75175567  0.90593248  0.92978172 ... -0.91135506 -0.46871177
  -0.54920564]
 [-0.45976742 -0.52550511 -0.48421007 ...  0.19247733  0.05901381
   0.41680288]]
[[-0.82004684 -1.02202571 -0.9916898  ...  0.87409641  0.26508427
   0.34316281]
 [-0.50936012 -0.46410379 -0.56941112 ...  0.32551494  0.16383972
   0.04702778]
 [-0.7548078  -0.95078365 -1.04911073 ...  1.38029879  1.01678554
   0.34322861]
 ...
 [ 1.27973353  1.42053356  1.60855361 ... -1.3015929  -0.95796902
  -0.24362728]
 [-0.49781492 -0.66962309 -0.85527186 ...  1.41306453  1.61953833
   0.85939177]
 [ 0.37072934  0.67304599  0.9513581  ... -1.22706772 -1.41055324
  -1.00995094]]


# Model

## Base

In [7]:
class Classification(ABC):
  @abstractmethod
  def fit(self, X: np.ndarray, y: np.ndarray) -> None:
    """
    Fit the model to the training data.
    """
    pass

  @abstractmethod
  def predict(self, X: np.ndarray) -> np.ndarray:
    """
    Predict the target values for the given input data.
    """
    pass

  @abstractmethod
  def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
    """
    Evaluate the model on the given input and target data.
    """
    pass

## Logistic Regression

In [8]:
class LogisticRegression(Classification):
  def __init__(
      self,
      random_state: int | None = SEED,
      solver: str | None = 'gradient_descent',
      learning_rate: float = LEARNING_RATE,
      max_iter: int = MAX_ITER,
  ):
      self.random_state = random_state
      self.solver = solver
      self.learning_rate = learning_rate
      self.max_iter = max_iter
      self.coef_ = None
      self.intercept_ = None
    
  def fit(self, X: np.ndarray, y: np.ndarray) -> None:
    raise NotImplementedError('Not implemented yet')

  def predict(self, X: np.ndarray) -> np.ndarray:
    raise NotImplementedError('Not implemented yet')

  def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
    raise NotImplementedError('Not implemented yet')
  

In [9]:
class LogisticRegression(Classification):
    def __init__(
        self,
        random_state: int | None = SEED,
        solver: str | None = 'gradient_descent',
        learning_rate: float = LEARNING_RATE,
        max_iter: int = MAX_ITER,
    ):
        self.random_state = random_state
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.weights = None
        self.bias = None

    def _softmax(self, z):
        # Subtracting max(z) for numerical stability (prevents overflow)
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def _one_hot(self, y, num_classes):
        return np.eye(num_classes)[y]

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Initialize Weights (Features x Classes) and Bias (1 x Classes)
        self.weights = np.zeros((n_features, n_classes))
        self.bias = np.zeros((1, n_classes))
        
        y_encoded = self._one_hot(y, n_classes)

        for i in range(self.max_iter):
            # Forward pass
            scores = np.dot(X, self.weights) + self.bias
            probs = self._softmax(scores)

            # Gradient calculation
            dw = (1 / n_samples) * np.dot(X.T, (probs - y_encoded))
            db = (1 / n_samples) * np.sum(probs - y_encoded, axis=0, keepdims=True)

            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            if i % 500 == 0:
                loss = -np.mean(np.sum(y_encoded * np.log(probs + 1e-15), axis=1))
                print(f"Iteration {i}: Loss {loss:.4f}")

    def predict(self, X: np.ndarray) -> np.ndarray:
        scores = np.dot(X, self.weights) + self.bias
        probs = self._softmax(scores)
        return np.argmax(probs, axis=1)

    def evaluate(self, y_hat: np.ndarray, y: np.ndarray) -> float:
        return np.mean(y_hat == y)

# Evaluation

In [10]:
# 1. Initialize the model
model = LogisticRegression()

# 2. Fit the model on scaled training data
# X_train_scaled and y_train were created in the previous preprocessing step
model.fit(X_train_scaled, y_train)

# 3. Make predictions on the test set
y_pred = model.predict(X_test_scaled)

# 4. Evaluate the performance
accuracy = model.evaluate(y_pred, y_test)

print(f"Model Accuracy: {accuracy * 100:.2f}%")

Iteration 0: Loss 1.9459
Iteration 500: Loss 0.7312
Iteration 1000: Loss 0.5625
Iteration 1500: Loss 0.4781
Iteration 2000: Loss 0.4268
Iteration 2500: Loss 0.3921
Iteration 3000: Loss 0.3671
Iteration 3500: Loss 0.3483
Iteration 4000: Loss 0.3336
Iteration 4500: Loss 0.3217
Iteration 5000: Loss 0.3120
Iteration 5500: Loss 0.3038
Iteration 6000: Loss 0.2969
Iteration 6500: Loss 0.2909
Iteration 7000: Loss 0.2857
Iteration 7500: Loss 0.2812
Iteration 8000: Loss 0.2772
Iteration 8500: Loss 0.2736
Iteration 9000: Loss 0.2703
Iteration 9500: Loss 0.2674
Model Accuracy: 92.25%
